# Train an Africa-wide Tiny U-Net and evaluate North Africa / Mediterranean

This notebook trains one lightweight, single-channel Landsat-8 thermal (B10) segmentation model. It deliberately trains on all African examples, while holding out North Africa/Mediterranean scenes for the regional result relevant to Tunisia.

**Before running:** extract the full 29 GB Africa archive in a location with at least 70 GB free space, then set `FULL_DATA_ROOT` below to the directory containing `patches/` and `masks/`. Run training on Colab/Kaggle or another CUDA machine; the local MX350/CPU environment is not appropriate for the full run.

The notebook does not download, move, or alter the source dataset.

In [ ]:
from pathlib import Path
import csv, random, re, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, WeightedRandomSampler

# In the repository use '..'. In Colab, clone/upload the repository then set this path.
PROJECT_ROOT = Path('..').resolve()
FULL_DATA_ROOT = Path(r'REPLACE_WITH_FULL_AFRICA_DATA_ROOT').expanduser()
METADATA_DIR = PROJECT_ROOT / 'data' / 'interim' / 'africa_metadata_extracted'
PATCH_INDEX = PROJECT_ROOT / 'data' / 'external' / 'activefire' / 'src' / 'train' / 'voting' / 'unet_16f_2conv_762' / 'dataset' / 'images_masks.csv'
OUTPUT_DIR = PROJECT_ROOT / 'reports' / 'training_africa_tiny_unet'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert FULL_DATA_ROOT.exists(), 'Set FULL_DATA_ROOT before continuing.'
assert METADATA_DIR.exists(), 'Extract africa_metadata.zip to data/interim/africa_metadata_extracted first.'
assert PATCH_INDEX.exists(), 'The ActiveFire reference index is missing.'

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
print('PyTorch:', torch.__version__)
if device.type == 'cuda': print('GPU:', torch.cuda.get_device_name(0))

## 1. Reuse the project preprocessing

Do **not** pre-convert GeoTIFFs into PNG/JPEG or normalize them once and save the result. `ActiveFireDataset` reads the original 16-bit GeoTIFFs and applies, per batch: B10 radiometric calibration (DN → brightness temperature), a fixed 270–420 K normalization window, and training-only geometric/thermal augmentation. This keeps the training input compatible with the eventual flight pipeline.

In [ ]:
import sys
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from fireedge.io.activefire import index_patches, read_mask
from fireedge.data.dataset import ActiveFireDataset

# The full archive must expose these paths. If not, point FULL_DATA_ROOT at its extracted inner folder.
df_all = pd.read_csv(STRICT_INVENTORY)

assert len(df_all) > 0, "No strict voting pairs were created."
assert df_all.image_path.map(Path.exists).all(), "Some image paths are missing."
assert df_all.mask_path.map(Path.exists).all(), "Some voting mask paths are missing."

df_all["path"] = pd.to_numeric(df_all["path"], errors="coerce")
df_all["row"] = pd.to_numeric(df_all["row"], errors="coerce")

print(f"Strict voting training pairs: {len(df_all):,}")
display(df_all.head())
df_all['path'] = pd.to_numeric(df_all['path'], errors='coerce')
df_all['row'] = pd.to_numeric(df_all['row'], errors='coerce')
assert len(df_all), 'No TIFF patches found. FULL_DATA_ROOT must contain patches/ or images/patches/.'
print(f'Indexed {len(df_all):,} image patches; paired masks: {df_all.has_mask.sum():,}')
display(df_all.head())

## 2. Attach source-scene coordinates and define the regional hold-out

A patch filename contains the Landsat WRS path/row. The small metadata archive maps each path/row to a scene footprint. North Africa/Mediterranean is defined by scene centre: 27–46°N, 18°W–36°E. This is reproducible and avoids depending on an informal folder name.

In [ ]:
PATCH_SCENE = re.compile(r'LC08_[A-Z0-9]+_(\d{3})(\d{3})_')
def mtl_number(text, key):
    m = re.search(rf'^\s*{key}\s*=\s*([-0-9.]+)', text, re.MULTILINE)
    if not m: raise ValueError(f'Missing {key}')
    return float(m.group(1))

scene_records = []
for archive in sorted(METADATA_DIR.glob('z*.zip')):
    with zipfile.ZipFile(archive) as zf:
        name = next(n for n in zf.namelist() if n.endswith('_MTL.txt'))
        text = zf.read(name).decode('utf-8', errors='replace')
    lats = [mtl_number(text, k) for k in ('CORNER_UL_LAT_PRODUCT','CORNER_UR_LAT_PRODUCT','CORNER_LL_LAT_PRODUCT','CORNER_LR_LAT_PRODUCT')]
    lons = [mtl_number(text, k) for k in ('CORNER_UL_LON_PRODUCT','CORNER_UR_LON_PRODUCT','CORNER_LL_LON_PRODUCT','CORNER_LR_LON_PRODUCT')]
    path = int(re.search(r'WRS_PATH\s*=\s*(\d+)', text).group(1))
    row = int(re.search(r'WRS_ROW\s*=\s*(\d+)', text).group(1))
    scene_records.append({'path': path, 'row': row, 'scene_lat': np.mean(lats), 'scene_lon': np.mean(lons),
                          'min_lat': min(lats), 'max_lat': max(lats), 'min_lon': min(lons), 'max_lon': max(lons)})
scene_df = pd.DataFrame(scene_records).drop_duplicates(['path','row'])

df = df_all.merge(scene_df, on=['path','row'], how='inner')
df['north_med'] = df.scene_lat.between(27, 46) & df.scene_lon.between(-18, 36)
# Correct parent scene id: do not use the patch-specific scene_id supplied by the current loader.
df['parent_scene'] = df.stem.str.replace(r'_p\d+$', '', regex=True)
print(f'African patches matched to coordinate metadata: {len(df):,}')
print('North Africa / Mediterranean:', int(df.north_med.sum()), 'patches from', df.loc[df.north_med, 'parent_scene'].nunique(), 'parent scenes')
display(df.groupby('north_med').agg(patches=('stem','size'), scenes=('parent_scene','nunique')).rename(index={False:'other_africa', True:'north_med'}))
df.to_csv(OUTPUT_DIR / 'africa_coordinate_inventory.csv', index=False)

## 3. Leak-free split

Reserve 20% of North Africa/Mediterranean scenes as the final regional test set. The remaining data are split by parent Landsat scene into training and validation. No patches from the same acquisition are allowed in more than one split.

In [ ]:
rng = np.random.default_rng(SEED)
north_scenes = df.loc[df.north_med, 'parent_scene'].drop_duplicates().to_numpy()
rng.shuffle(north_scenes)
n_north_test = max(1, int(round(0.20 * len(north_scenes))))
north_test_scenes = set(north_scenes[:n_north_test])
test_df = df[df.parent_scene.isin(north_test_scenes)].copy()
pool_df = df[~df.parent_scene.isin(north_test_scenes)].copy()

pool_scenes = pool_df.parent_scene.drop_duplicates().to_numpy()
rng.shuffle(pool_scenes)
n_val = max(1, int(round(0.15 * len(pool_scenes))))
val_scenes = set(pool_scenes[:n_val])
val_df = pool_df[pool_df.parent_scene.isin(val_scenes)].copy()
train_df = pool_df[~pool_df.parent_scene.isin(val_scenes)].copy()

for name, part in [('train', train_df), ('validation', val_df), ('north_med_test', test_df)]:
    print(f'{name:15s} patches={len(part):,} scenes={part.parent_scene.nunique():,} north_med={part.north_med.sum():,}')
    part.to_csv(OUTPUT_DIR / f'{name}_inventory.csv', index=False)
assert not (set(train_df.parent_scene) & set(val_df.parent_scene))
assert not (set(train_df.parent_scene) & set(test_df.parent_scene))
assert not (set(val_df.parent_scene) & set(test_df.parent_scene))

## 4. Audit labels before training

This prevents a misleading experiment. The current `voting` labels are consensus algorithm masks, not manual masks. Report them as pseudo-labels unless you replace `mask_path` with the manual annotation files. Also inspect how many all-zero masks are available: pixel-level background exists in every positive patch, but all-zero scenes are needed for a strong false-alarm test.

In [ ]:
def fire_pixels(row):
    return int(read_mask(row.mask_path).sum())
audit_sample = train_df.sample(min(2000, len(train_df)), random_state=SEED).copy()
audit_sample['fire_pixels'] = audit_sample.apply(fire_pixels, axis=1)
display(audit_sample.fire_pixels.describe(percentiles=[.1,.25,.5,.75,.9,.99]).to_frame().T)
print('All-zero masks in audit:', int((audit_sample.fire_pixels == 0).sum()), '/', len(audit_sample))
assert (audit_sample.fire_pixels > 0).any(), 'No positive labels were found; stop and check mask pairing.'
# If all-zero scenes are scarce, add genuine non-fire Landsat patches before claiming a false-alarm rate.

## 5. Dataset, regional oversampling, and Tiny U-Net

Fixed-window normalization preserves meaningful brightness-temperature differences. North Africa/Mediterranean samples are oversampled 3× in training, but are never duplicated into validation or test.

In [ ]:
BATCH_SIZE = 8 if device.type == 'cuda' else 2
NUM_WORKERS = 2 if device.type == 'cuda' else 0
train_ds = ActiveFireDataset(train_df, channels=(8,), norm_method='fixed', augment=True, random_seed=SEED)
val_ds = ActiveFireDataset(val_df, channels=(8,), norm_method='fixed', augment=False)
test_ds = ActiveFireDataset(test_df, channels=(8,), norm_method='fixed', augment=False)
weights = np.where(train_df.north_med.to_numpy(), 3.0, 1.0)
sampler = WeightedRandomSampler(torch.as_tensor(weights, dtype=torch.double), num_samples=len(train_ds), replacement=True)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=NUM_WORKERS, pin_memory=device.type=='cuda')
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=device.type=='cuda')
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=device.type=='cuda')
x, y = next(iter(train_loader)); print(x.shape, x.dtype, (x.min().item(), x.max().item()), y.unique())

class ConvBlock(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.block = nn.Sequential(nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
                                   nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True))
    def forward(self, x): return self.block(x)

class TinyUNet(nn.Module):
    def __init__(self, base=8):
        super().__init__()
        self.e1, self.e2, self.e3 = ConvBlock(1, base), ConvBlock(base, base*2), ConvBlock(base*2, base*4)
        self.pool = nn.MaxPool2d(2)
        self.mid = ConvBlock(base*4, base*8)
        self.u3 = nn.ConvTranspose2d(base*8, base*4, 2, 2); self.d3 = ConvBlock(base*8, base*4)
        self.u2 = nn.ConvTranspose2d(base*4, base*2, 2, 2); self.d2 = ConvBlock(base*4, base*2)
        self.u1 = nn.ConvTranspose2d(base*2, base, 2, 2); self.d1 = ConvBlock(base*2, base)
        self.out = nn.Conv2d(base, 1, 1)
    def forward(self, x):
        e1=self.e1(x); e2=self.e2(self.pool(e1)); e3=self.e3(self.pool(e2)); m=self.mid(self.pool(e3))
        d3=self.d3(torch.cat([self.u3(m),e3],1)); d2=self.d2(torch.cat([self.u2(d3),e2],1)); d1=self.d1(torch.cat([self.u1(d2),e1],1))
        return self.out(d1)

model = TinyUNet().to(device)
print('Parameters:', sum(p.numel() for p in model.parameters()))

In [ ]:
class FocalTverskyLoss(nn.Module):
    def __init__(self, alpha=0.3, beta=0.7, gamma=0.75, eps=1e-6):
        super().__init__(); self.alpha, self.beta, self.gamma, self.eps = alpha, beta, gamma, eps
    def forward(self, logits, target):
        p = torch.sigmoid(logits); dims = (1,2,3)
        tp=(p*target).sum(dims); fp=(p*(1-target)).sum(dims); fn=((1-p)*target).sum(dims)
        tversky=(tp+self.eps)/(tp+self.alpha*fp+self.beta*fn+self.eps)
        return ((1-tversky)**self.gamma).mean()

bce = nn.BCEWithLogitsLoss()
ft = FocalTverskyLoss()
def loss_fn(logits, targets): return 0.3*bce(logits, targets) + 0.7*ft(logits, targets)

@torch.no_grad()
def evaluate(loader, threshold=0.20):
    model.eval(); tp=fp=fn=0; losses=[]
    for images, masks in loader:
        images, masks = images.to(device), masks.to(device)
        logits = model(images); losses.append(loss_fn(logits, masks).item())
        pred = torch.sigmoid(logits) >= threshold; target = masks.bool()
        tp += (pred & target).sum().item(); fp += (pred & ~target).sum().item(); fn += (~pred & target).sum().item()
    recall=tp/(tp+fn+1e-9); precision=tp/(tp+fp+1e-9); iou=tp/(tp+fp+fn+1e-9); dice=2*tp/(2*tp+fp+fn+1e-9)
    return {'loss':float(np.mean(losses)), 'recall':recall, 'precision':precision, 'iou':iou, 'dice':dice, 'tp':tp, 'fp':fp, 'fn':fn}

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
EPOCHS, PATIENCE = 30, 6
history, best_dice, stale = [], -1.0, 0

## 6. Train and select the checkpoint by validation Dice

The threshold is intentionally low initially (0.20) because missed fire is worse than a reviewable false alarm. It is tuned on validation data after training, never on the held-out regional test set.

In [ ]:
for epoch in range(1, EPOCHS+1):
    model.train(); train_losses=[]
    for images, masks in train_loader:
        images, masks = images.to(device), masks.to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(images); loss = loss_fn(logits, masks)
        loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); optimizer.step()
        train_losses.append(loss.item())
    metrics = evaluate(val_loader, threshold=0.20)
    row = {'epoch':epoch, 'train_loss':float(np.mean(train_losses)), **metrics}; history.append(row)
    print(f"{epoch:02d} train={row['train_loss']:.4f} val_loss={row['loss']:.4f} recall={row['recall']:.3f} precision={row['precision']:.3f} dice={row['dice']:.3f} iou={row['iou']:.3f}")
    if metrics['dice'] > best_dice:
        best_dice, stale = metrics['dice'], 0
        torch.save({'model_state':model.state_dict(), 'epoch':epoch, 'validation':metrics, 'threshold':0.20}, OUTPUT_DIR/'best_tiny_unet.pt')
    else:
        stale += 1
        if stale >= PATIENCE:
            print('Early stopping'); break
history_df=pd.DataFrame(history); history_df.to_csv(OUTPUT_DIR/'history.csv', index=False)
display(history_df.tail())

In [ ]:
checkpoint=torch.load(OUTPUT_DIR/'best_tiny_unet.pt', map_location=device, weights_only=False)
model.load_state_dict(checkpoint['model_state'])
threshold_grid=np.arange(0.05, 0.51, 0.05)
threshold_results=pd.DataFrame([{'threshold':float(t), **evaluate(val_loader, threshold=float(t))} for t in threshold_grid])
# Recall-prioritized selection: among thresholds retaining >=95% of maximum recall, take best Dice.
eligible=threshold_results[threshold_results.recall >= 0.95*threshold_results.recall.max()]
chosen=eligible.sort_values(['dice','precision'], ascending=False).iloc[0]
threshold_results.to_csv(OUTPUT_DIR/'validation_threshold_sweep.csv', index=False)
print('Chosen threshold:', float(chosen.threshold)); display(threshold_results)

regional_test=evaluate(test_loader, threshold=float(chosen.threshold))
pd.DataFrame([regional_test]).to_csv(OUTPUT_DIR/'north_med_test_metrics.csv', index=False)
print('HELD-OUT NORTH AFRICA/MEDITERRANEAN TEST METRICS')
print(regional_test)
assert regional_test['tp'] + regional_test['fn'] > 0, 'Test set has no labelled fire pixels; report this limitation.'

In [ ]:
# Visual quality check: original normalized thermal input, label, probability, and thresholded output.
model.eval(); images, masks=next(iter(test_loader)); images=images.to(device)
with torch.no_grad(): probs=torch.sigmoid(model(images)).cpu()
n=min(4, len(images)); fig, axes=plt.subplots(n,4,figsize=(14,3*n))
for i in range(n):
    axes[i,0].imshow(images[i,0].cpu(), cmap='inferno', vmin=0, vmax=1); axes[i,0].set_title('B10 normalized')
    axes[i,1].imshow(masks[i,0], cmap='gray'); axes[i,1].set_title('Label')
    axes[i,2].imshow(probs[i,0], cmap='magma', vmin=0, vmax=1); axes[i,2].set_title('Fire probability')
    axes[i,3].imshow(probs[i,0] >= float(chosen.threshold), cmap='gray'); axes[i,3].set_title('Prediction')
    for ax in axes[i]: ax.axis('off')
plt.tight_layout(); plt.savefig(OUTPUT_DIR/'north_med_predictions.png', dpi=160); plt.show()

## What to report

Report the held-out North Africa/Mediterranean recall, precision, Dice/F1, IoU, selected threshold, number of held-out scenes, and the fact that labels are voting/consensus masks unless replaced by manual annotations. Do not report pixel accuracy: it is dominated by background pixels.

Before flight export, run a separate robustness test using `MicrobolometerDegrader` and then export the chosen checkpoint to ONNX. Do not mix that sensor-degradation experiment into this baseline run.